Estimativa de tamanho na GPU

In [ ]:
model_size = 4
Q_size = 16

m = 1.2*(model_size*4)/(32/Q_size)

m

9.6

## Inicializando modelo

## Classe LLM


### Llama-cpp

In [1]:
from utils import langchain_to_jinja, jinja_to_langchain
from langchain_core.messages import BaseMessage, ToolMessage, AIMessage, SystemMessage, HumanMessage
from llama_cpp import Llama
class LLML(Llama):
    
    def __init__(self, *arg, **kwargs):
        #self.model_config = read_yaml()
        super().__init__(*arg, **kwargs)

    def call(self, messages:list[dict], tools:list[dict]=[]):
        response = self.create_chat_completion(
            messages = messages,
            tools=tools
        )
        return response

    def invoke(self, messages: list[BaseMessage], tools: list[ToolMessage]=[]):
        jinja_messages = langchain_to_jinja(messages)
        ai_message= self.call(messages=jinja_messages, tools=tools)['choices'][0]['message']['content']
        langchain_message =jinja_to_langchain(ai_message, think=None)
        return langchain_message

/home/migueldcarvalho/miniforge3/envs/geral/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


In [2]:
chat = LLML(
    model_path="/home/migueldcarvalho/Projetos/Chatbot/modelos_locais/Qwen3-4B-Instruct-2507-UD-Q4_K_XL.gguf", 
    n_gpu_layers=-1, 
    verbose=False,
    n_ctx=1024
    )

llama_context: n_ctx_per_seq (1024) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


### Transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/migueldcarvalho/miniforge3/envs/chatbot/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [ ]:
models = ["Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen3-0.6B", "Qwen/Qwen2.5-Coder-0.5B-Instruct", "Qwen/Qwen3-4B-Instruct-2507"]
model_name = models[3]

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto",
    trust_remote_code=True # Recomendado para modelos Qwen/comunidade
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [ ]:
from Chatbot.utils import jinja_to_langchain, langchain_to_jinja, read_yaml
import torch
import gc

class LLM():
 
    def __init__(self, max_steps=5, history_name="history", **kwargs):

    # ============ Models ==================
        # LLM

        self.load_config()

        self.tokenizer = kwargs.get('tokenizer')
        self.model = kwargs.get('model')
        # Embedding model         
        #self.client = kwargs.get('client')
        #self.encoder = kwargs.get('encoder')

    # ============ System prompts ==================
        #self.system_prompt = read_yaml("config/system_prompt.yaml")
        #self.tools_doc = read_yaml("config/tools_doc.yaml")
    
    def load_config(self):
        self.model_config = read_yaml("../config/model_config.yaml") # "config/model_config.yaml" no colab
        
    #@measure_time
    def call(self, messages: list, tools: list=[]) -> tuple:

        self.load_config()

        text = self.tokenizer.apply_chat_template(
            messages, 
            tools=tools, # lista de dicionarios contendo informações de cada tool, segundo padrão da openai
            tokenize=False,
            **self.model_config[self.model.config.name_or_path]['template']
        )

        
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        # conduct text completion
        with torch.no_grad():
            generated_ids = self.model.generate(
                **model_inputs,
                **self.model_config[self.model.config.name_or_path]['generate']
            )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
        
        try:
            # rindex finding 151668 (</think>)
            index = len(output_ids) - output_ids[::-1].index(151668)
        except ValueError:
            index = 0

        thinking_content = self.tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
        content = self.tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

        gc.collect()
        torch.cuda.empty_cache()
        
        return content, thinking_content
    
    def invoke(self, messages: list, tools: list=[]) -> tuple:
        jinja_messages = langchain_to_jinja(messages)
        ai_message, think = self.call(messages=jinja_messages, tools=tools)
        langchain_message =jinja_to_langchain(ai_message, think)

        return langchain_message

In [16]:
chat = LLM(model=model, tokenizer=tokenizer)

In [4]:
from chatbot.utils import langchain_to_jinja

In [1]:
from config.paths import SYSTEM_PROMPTS

## Testes

In [9]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from utils import measure_time, read_yaml
from config.paths import SYSTEM_PROMPTS
from pandasql import sqldf
from pathlib import Path
import pandas as pd
import requests
import chardet
import yaml
import uuid
import torch
import re
import os

def write_yaml(path: str, data: dict, overwrite: bool=False)->None:
    def str_presenter(dumper, data):
        if '\n' in data:
            # Usa o estilo '|' (literal block) se encontrar uma quebra de linha
            return dumper.represent_scalar('tag:yaml.org,2002:str', data, style='|')
        return dumper.represent_scalar('tag:yaml.org,2002:str', data)

    # 2. Adicionamos essa regra ao Dumper padrão e ao SafeDumper
    yaml.add_representer(str, str_presenter)
    yaml.SafeDumper.add_representer(str, str_presenter)
    
    if Path(path).exists() and not overwrite:
        with open(path, "r") as f:
            existing_data = yaml.safe_load(f)
        if existing_data:
            print(existing_data)
            data = {**existing_data, **data}
    
    with open(path, "w", encoding='utf-8') as f:
        yaml.dump(data, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

class GovPath():
    def __init__(self, link: str):
        self.basename = Path(os.path.basename(link)).stem
        self.gov_repo = Path("/home/migueldcarvalho/Projetos/Chatbot/GovData")
        
        self.dirname = Path(os.path.basename(link)).stem
        self.dirpath = self.gov_repo/self.basename

        self.analysisname = Path(str(Path(self.basename)) + '_analysis.yaml')
        self.analysispath = self.gov_repo/self.dirname/self.analysisname
        
        self.dataname = os.path.basename(link)
        self.datapath = self.gov_repo/self.dirname/self.dataname
         
def log_tabular_queries(filename:str, logdata: dict):
    path = "/home/migueldcarvalho/Projetos/Chatbot/Testes/LogSQLqueries.yaml"
    
    log_history = read_yaml(path)
    file_history=None

    if log_history:
        file_history = log_history.get(filename)
    print(file_history)
            
    if file_history:
        updated_history = {filename: {str(uuid.uuid4()): logdata, **file_history}} # Atualizando
        #log_history.update({filename: file_history})
    else:
        updated_history = {filename: {str(uuid.uuid4()): logdata}}
    
    print(file_history)
    write_yaml(path=path, data=updated_history, overwrite=False)


def user_query(user_question: str, filepath)->HumanMessage:

    #filename = Path(str(Path(os.path.basename(link)).stem) + '_analysis.yaml')
    #dirname = Path(os.path.basename(link)).stem

    metadata = read_yaml(path=filepath)

    dataset_description = ""
    rows = metadata['shape'][0]
    columns = metadata['shape'][1]
    column_type_dictionary = metadata['dtypes']
    sample_data = metadata['sample_data'][:3]
    basic_statistical_analysis = ""

    input= str(f'''   
    <METADATA>
    Description: {dataset_description}
    Dimensions: {rows} rows x {columns} columns
    Schema (Columns and Data Types): {column_type_dictionary}
    Sample_data: {sample_data}
    Statistical Summary: {basic_statistical_analysis}
    </METADATA>

    <USER_QUESTION>
    {user_question}
    </USER_QUESTION>
    ''')

    return HumanMessage(content=input)

def log_events(data:dict, filepath: str, format:str='yaml')->None:
    if format=='yaml':
        write_yaml(data=data, path=filepath, overwrite=False)

def encoding_type(file_path):
    with open(file_path, 'rb') as f:
        raw_data = f.read(10000)
        encoding = chardet.detect(raw_data)['encoding']
    return encoding

gov_repo = Path("/home/migueldcarvalho/Projetos/Chatbot/GovData")
def download_resource(link: str):
    file_name = os.path.basename(link)
    dir = Path(file_name).stem

    destination = gov_repo / dir
    destination.mkdir(parents=True, exist_ok=True)

    
    response = requests.get(link)
    if response.status_code != 200:
        return response.status_code
    

    file_path = destination/file_name

    with open(file_path, 'wb') as f:
        f.write(response.content)

    return response.status_code

def pre_analysis(filepath: str, data)->None:
    metadata = {
    "success": True,
    # Descreve cada coluna identificando valores de mínimo, máximo e desvio padrão (numeros), 
    # tipos mais frequentes, frequencia dos tipos mais frequentes e tipos unicos (texto)
    "describe": data.describe().to_dict(),
    "kurtosis": data.select_dtypes(include=['number']).kurtosis().to_dict(),
    "skewness": data.select_dtypes(include=['number']).skew().to_dict(),
    "correlation_matrix": data.select_dtypes(include=['number']).corr().to_dict(),
    "shape": list(data.shape), # dimensões da matriz
    "columns": data.columns.tolist(), # colunas
    "dtypes": data.dtypes.astype(str).to_dict(), # tipos de cada coluna
    "sample_data": data.head().to_dict('records'), # Cinco primeiros valores
    "null_counts": data.isnull().sum().to_dict(), # contagem de células vazias
    "memory_usage": float(data.memory_usage(deep=True).sum()/(1024**2)) # Quantidade de memoria RAM em Mb
    }

    write_yaml(path=filepath, data=metadata, overwrite=True)

    return metadata

def generate_query(query: HumanMessage)->str:
    system_prompt = read_yaml(SYSTEM_PROMPTS).get('system_prompt_queries').get('sql1')
    system_message = SystemMessage(content=system_prompt)

    response = chat.invoke([system_message, query])

    #code = re.sub(r"</?code>", "", response.content).strip() # retira tags

    return response

def tabular_analysis(query: str, link: str):
    paths = GovPath(link)

    if not Path(paths.datapath).exists():
        status = download_resource(link)
        if status == 200:
            print("Download concluído")
        else:
            return {"Erro de requisição": status}
        
    dt_table = pd.read_csv(paths.datapath, encoding=encoding_type(paths.datapath), on_bad_lines='skip', sep=';', decimal=',') #on_bad_lines='warn'
    
    pre_analysis(filepath=paths.analysispath, data=dt_table)
    print("pre-análise concluida...")

    query = user_query(query, paths.analysispath)

    response, inferencetime = measure_time(generate_query)(query)

    dt_table = pd.read_csv(paths.datapath, encoding=encoding_type(paths.datapath), on_bad_lines='skip', sep=';', decimal=',') #on_bad_lines='warn'
    
    error_type = "No error"
    if '<code>' in response.content:
        sql_query = re.sub(r"</?code>", "", response.content).strip() # retira tags
        try:
            result = sqldf(sql_query, {'dt_table': dt_table})
        except Exception as e:
            error_type = str(e)
            result = str(e)
        print(sql_query)
    else:
        result = response.content

    log_tabular_queries(
        logdata={'UserQuery': query.content, 'LLMResponse': response.content, 
               'InferenceTime': inferencetime, 'Device':torch.cuda.get_device_name(0), 'Error': error_type},
        filename=paths.basename,)

    return response.content, result

## Código em SQL

Problemas com geração: /home/migueldcarvalho/Projetos/Chatbot/Testes/sql_problemas.md

### Testes

In [16]:
from pandasql import sqldf
import pandas as pd

links = {
'ifpa': "https://pda.ifpa.edu.br/dataset/b72101b4-9d52-4a23-9ca4-09bc32413e5e/resource/13e61895-e0b7-4061-9759-671459f04354/download/pda_chamados_ifpa-2020.csv",
'multas': "http://dadosabertos.ibama.gov.br/dados/SICAFI/AC/Quantidade/multasDistribuidasBensTutelados.csv",
'policecalls': "https://dados.fortaleza.ce.gov.br/dataset/5fd98aaa-1e16-425a-9e42-2c580d5d156c/resource/e65d44e1-8cc4-4dd4-addc-eee741845c94/download/policecalls.csv",
'despesa_covid': "https://dadosabertos.rj.gov.br/dataset/0135efb4-be5c-451b-8930-7283d4a0f447/resource/7242cc7b-b504-467c-b37b-3b6eaf0ae339/download/despesas-pandemia.csv",
'funcionário_tercerizados2022': "https://dados.df.gov.br/dataset/4cd9832d-959d-405e-8e21-69c40187195d/resource/fbf987e6-982b-4765-94cd-817144e5ace3/download/funcionariosterceirizados2022.csv",
}

link = links['funcionário_tercerizados2022']
paths = GovPath(link)

query = "há quanto funciários na empresa REAL JG SERVIÇOS GERAIS"

def query_tab(query:str, link:str):
  paths = GovPath(link)

  response, result = tabular_analysis(query=query, link=link)

  return response, result

response, result = query_tab(query, link)
print(response)
result

pre-análise concluida...
SELECT COUNT(*) AS quantidade_funcionarios
FROM dt_table
WHERE Empresa = 'REAL JG SERVIÇOS GERAIS'
{'62c240c0-46f1-4f34-b370-26fc73ab9e9b': {'UserQuery': "   \n    <METADATA>\n    Description: \n    Dimensions: 65782 rows x 4 columns\n    Schema (Columns and Data Types): {'Terceirizado': 'str', 'Cargo': 'str', 'Empresa': 'str', 'Mês/Ano': 'str'}\n    Sample_data: [{'Terceirizado': 'ABADIA ANTUNES MIRANDA', 'Cargo': 'SERVENTE', 'Empresa': 'SOLUÇÕES SERVIÇOS TERCEIRIZADOS EIRELI?', 'Mês/Ano': '01/01/2022'}, {'Terceirizado': 'ABEDENAQUE ALVES DE OLIVEIRA JUNIOR', 'Cargo': 'VIGILANTE', 'Empresa': 'BRASFORT EMPRESA DE SEGURANÇA LTDA', 'Mês/Ano': '01/01/2022'}, {'Terceirizado': 'ABILIO RODRIGUES DE QUEIROZ', 'Cargo': 'SERVENTE', 'Empresa': 'REAL JG SERVIÇOS GERAIS', 'Mês/Ano': '01/01/2022'}]\n    Statistical Summary: \n    </METADATA>\n\n    <USER_QUESTION>\n    Quero nomes, funções de admissões dos funcionarios de REAL JG SERVIÇOS GERAIS\n    </USER_QUESTION>\n    "

,quantidade_funcionarios
0,7029


In [11]:
result

"Execution failed on sql 'SELECT \n    Terceirizado, \n    Cargo, \n    Mês/Ano \nFROM \n    dt_table \nWHERE \n    Empresa = 'REAL JG SERVIÇOS GERAIS'': (sqlite3.OperationalError) no such column: Mês\n[SQL: SELECT \n    Terceirizado, \n    Cargo, \n    Mês/Ano \nFROM \n    dt_table \nWHERE \n    Empresa = 'REAL JG SERVIÇOS GERAIS']\n(Background on this error at: https://sqlalche.me/e/20/e3q8)"

In [13]:
from pandasql import sqldf
import pandas as pd

sql_query ='''

SELECT 
    Terceirizado, 
    Cargo, 
    'Mês/Ano' 
FROM 
    dt_table 
WHERE 
    Empresa = 'REAL JG SERVIÇOS GERAIS'
'''

dt_table = pd.read_csv(paths.datapath, encoding=encoding_type(paths.datapath), on_bad_lines='skip', sep=';', decimal=',')
res = sqldf(sql_query, {'dt_table': dt_table})
res

,Terceirizado,Cargo,'Mês/Ano'
0,ABILIO RODRIGUES DE QUEIROZ,SERVENTE,Mês/Ano
1,ADALBERTO BORGES DOS SANTOS,SERVENTE,Mês/Ano
2,ADELINA NERES DO NASCIMENTO,SERVENTE,Mês/Ano
3,ADILIA PATRICIA DA SILVA ROSA,SERVENTE,Mês/Ano
4,ADILSON GALVAO DA SILVA,SERVENTE,Mês/Ano
...,...,...,...
7024,WIVIANE CHAGAS LIMA,SERVENTE,Mês/Ano
7025,WYLLIANE LORRANNE AMORIM DE SOUZA,SERVENTE,Mês/Ano
7026,ZAIRA BARBOSA DINIZ,SERVENTE,Mês/Ano
7027,ZELIA RIBEIRO DA SILVA,SERVENTE,Mês/Ano


In [71]:
paths = GovPath(link)
df = pd.read_csv(paths.datapath, encoding=encoding_type(paths.datapath), on_bad_lines='skip', sep=';', decimal=',') #on_bad_lines='warn'

In [73]:
result = df[df['Requerente'] == 'MARCONDES AZEVEDO'].copy()
result

,Chamado - ID,Categoria,Título,Requerente,Aberto,Fechado
1,14525,Sistemas > SIGAA - TURMAS,CHAMADO : SISTEMAS => SIG_TUR => SIGAA - TURMA,MARCONDES AZEVEDO,30-12-2020 11:52:01,30-12-2020 14:55:49
2,14524,Sistemas > SIGAA - RELATORIOS,CHAMADO : SISTEMAS => SIG_REL => SIGAA - RELAT,MARCONDES AZEVEDO,30-12-2020 11:51:01,30-12-2020 14:55:56
3,14523,Sistemas > SIGAA - RELATORIOS,CHAMADO : SISTEMAS => SIG_REL => SIGAA - RELAT,MARCONDES AZEVEDO,30-12-2020 11:48:01,04-01-2021 15:20:52
4,14522,Sistemas > SIGAA - MANUTENCAO/ALTERACOES/AJUSTES,CHAMADO : SISTEMAS => SIG_MAN => SIGAA - MANUT,MARCONDES AZEVEDO,30-12-2020 11:47:02,04-01-2021 15:20:42
5,14521,Sistemas > SIGAA - MANUTENCAO/ALTERACOES/AJUSTES,CHAMADO : SISTEMAS => SIG_MAN => SIGAA - MANUT,MARCONDES AZEVEDO,30-12-2020 11:44:01,31-12-2020 09:01:47
6,14520,Sistemas > SIPAC - OUTROS,CHAMADO : SISTEMAS => PAC_ZUT => SIPAC - OUTRO,MARCONDES AZEVEDO,30-12-2020 11:43:01,30-12-2020 14:56:02
7,14519,Sistemas > SIPAC - RELATORIOS,CHAMADO : SISTEMAS => PAC_REL => SIPAC - RELAT,MARCONDES AZEVEDO,30-12-2020 11:42:01,30-12-2020 14:56:10
8,14518,Sistemas > SIPAC - PERFIL/TIPO DE ACESSO,CHAMADO : SISTEMAS => PAC_PRF => SIPAC - PERFI,MARCONDES AZEVEDO,30-12-2020 11:40:01,04-01-2021 15:20:26
9,14517,Sistemas > SIPAC - MANUTENCAO/ALTERACOES/AJUSTES,CHAMADO : SISTEMAS => PAC_MAN => SIPAC - MANUT,MARCONDES AZEVEDO,30-12-2020 11:39:01,30-12-2020 15:05:26
10,14516,Sistemas > SIPAC - ADICIONAR UNIDADE DE TRABALHO,CHAMADO : SISTEMAS => PAC_ADD => SIPAC - ADICI,MARCONDES AZEVEDO,30-12-2020 11:36:02,04-01-2021 15:20:32


In [29]:
import torch

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("GPU não encontrada")


NVIDIA GeForce RTX 4050 Laptop GPU


In [ ]:
def user_query(user_question: str, link: str)->HumanMessage:

    filename = Path(str(Path(os.path.basename(link)).stem) + '_analysis.yaml')
    dirname = Path(os.path.basename(link)).stem

    metadata = read_yaml(path=gov_repo/dirname/filename)

    dataset_description = ""
    rows = metadata['shape'][0]
    columns = metadata['shape'][1]
    column_type_dictionary = metadata['dtypes']
    sample_data = metadata['sample_data'][:3]
    basic_statistical_analysis = ""

    input= f'''   
    <METADATA>
    Description: {dataset_description}
    Dimensions: {rows} rows x {columns} columns
    Schema (Columns and Data Types): {column_type_dictionary}
    Sample_data: {sample_data}
    Statistical Summary: {basic_statistical_analysis}
    </METADATA>

    <USER_QUESTION>
    {user_question}
    </USER_QUESTION>
    '''

    return HumanMessage(content=input)

query = "teste"

print(user_query(query).content)

   
    <METADATA>
    Description: 
    Dimensions: 3104 rows x 6 columns
    Schema (Columns and Data Types): {'Aberto': 'object', 'Categoria': 'object', 'Chamado - ID': 'int64', 'Fechado': 'object', 'Requerente': 'object', 'Título': 'object'}
    Sample_data: [{'Aberto': '30-12-2020 16:52:01', 'Categoria': 'Sistemas > SIGAA - PERFIL USUARIO', 'Chamado - ID': 14526, 'Fechado': '08-01-2021 12:29:54', 'Requerente': 'HELIO FILHO', 'Título': 'CHAMADO :   SISTEMAS => SIG_PER =>  SIGAA - PERFI'}, {'Aberto': '30-12-2020 11:52:01', 'Categoria': 'Sistemas > SIGAA - TURMAS', 'Chamado - ID': 14525, 'Fechado': '30-12-2020 14:55:49', 'Requerente': 'MARCONDES AZEVEDO', 'Título': 'CHAMADO :   SISTEMAS => SIG_TUR =>  SIGAA - TURMA'}, {'Aberto': '30-12-2020 11:51:01', 'Categoria': 'Sistemas > SIGAA - RELATORIOS', 'Chamado - ID': 14524, 'Fechado': '30-12-2020 14:55:56', 'Requerente': 'MARCONDES AZEVEDO', 'Título': 'CHAMADO :   SISTEMAS => SIG_REL =>  SIGAA - RELAT'}]
    Statistical Summary: 
    </ME